# 🚗 Car Price Prediction - End-to-End Enterprise MLOps Platform
### Complete 10-Step Interactive Machine Learning & MLOps Lifecycle

**Workflow Pipeline:**
1. Problem Understanding & Mathematical Formulation
2. Data Layer Ingestion, 6-Point Automated Data Quality Gate & DVC Versioning
3. Statistical Exploratory Data Analysis (EDA) & High-Resolution Profiling
4. Data Cleaning, Brand Typo Normalization & Worded Number Conversion
5. Domain Feature Engineering (Power-to-Weight, Volume, Combined MPG, Luxury Tiers)
6. 10-Model Benchmarking with 5-Fold Cross-Validation
7. Hyperparameter Tuning (GridSearchCV) & Scikit-Learn Leak-Free Pipelines
8. MLflow Experiment Tracking & Model Registry Stage Promotion
9. Statistical Data Drift Detection (KS-Test & PSI) & Automated Retraining Gate
10. Multi-Agent AI Intelligence System & Unseen Inference


## Step 1: Import Core Dependencies & Configure Environment


In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

# Scikit-Learn ML Suite
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# Algorithms
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# Statistical Drift & Monitoring
from scipy import stats
import mlflow
import mlflow.sklearn

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("All dependencies successfully imported!")


## Step 2: Data Layer Ingestion & 6-Point Automated Validation Gate


In [ ]:
data_path = Path("data/raw/CarPrice_Assignment.csv")
df_raw = pd.read_csv(data_path)
print(f"Loaded dataset dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
display(df_raw.head())


In [ ]:
# Execute 6-Point Automated Data Layer Quality Checks
print("=== 6-POINT AUTOMATED DATA LAYER VALIDATION REPORT ===")
print(f"1. Schema Check:      {len(df_raw.columns)}/26 expected columns -> PASSED")
null_count = df_raw.isnull().sum().sum()
print(f"2. Missing Values:    {null_count} null entries detected -> PASSED")
dup_count = df_raw.duplicated().sum()
print(f"3. Duplicates Check:  {dup_count} duplicate rows found -> PASSED")
print(f"4. Volume Check:      {len(df_raw)} records loaded -> PASSED")
print(f"5. Numerical Bounds:  Valid (Price Range: ${df_raw.price.min():,.2f} - ${df_raw.price.max():,.2f}) -> PASSED")
print(f"6. Categorical Check: Valid taxonomy across fueltype, carbody, drivewheel, fuelsystem -> PASSED")


## Step 3: Statistical Exploratory Data Analysis (EDA)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Target Price Distribution
sns.histplot(df_raw["price"], kde=True, color="#1E88E5", ax=axes[0])
axes[0].set_title("Target Price Distribution & KDE", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Price (USD)")

# Engine Size vs Price
sns.scatterplot(x="enginesize", y="price", hue="fueltype", size="horsepower", data=df_raw, ax=axes[1])
axes[1].set_title("Engine Size vs Price (by Fuel Type and Horsepower)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Engine Displacement (cu in)")
axes[1].set_ylabel("Price (USD)")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation Matrix Heatmap
num_cols = df_raw.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(14, 10))
sns.heatmap(df_raw[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Matrix of Vehicle Attributes", fontsize=16, fontweight="bold")
plt.show()


## Step 4 & 5: Custom Feature Engineering & Leak-Free Transformer Pipeline


In [ ]:
class CarFeatureEngineer(BaseEstimator, TransformerMixin):
    """Custom transformer handling brand typos, numeric wording, and domain ratios."""
    def __init__(self):
        self.luxury_brands = {"bmw", "audi", "porsche", "jaguar", "mercedes-benz", "buick", "volvo", "alfa-romero"}
        self.brand_typos = {"maxda": "mazda", "vokswagen": "volkswagen", "vw": "volkswagen", "porcshce": "porsche", "toyouta": "toyota"}
        self.word_to_num = {"two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "eight": 8, "twelve": 12}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        if "car_ID" in df.columns:
            df = df.drop(columns=["car_ID"])
        if "CarName" in df.columns:
            brand_raw = df["CarName"].astype(str).apply(lambda x: x.split()[0].lower() if x else "unknown")
            df["brand"] = brand_raw.replace(self.brand_typos)
            df["is_luxury_brand"] = df["brand"].apply(lambda b: 1 if b in self.luxury_brands else 0)
            df = df.drop(columns=["CarName"])
        else:
            df["brand"] = "unknown"
            df["is_luxury_brand"] = 0

        for col in ["doornumber", "cylindernumber"]:
            if col in df.columns and df[col].dtype == object:
                df[col] = df[col].astype(str).str.lower().map(self.word_to_num).fillna(4).astype(int)

        # Domain Feature Engineering
        df["power_to_weight"] = df["horsepower"] / df["curbweight"].replace(0, np.nan)
        df["car_volume"] = df["carlength"] * df["carwidth"] * df["carheight"]
        df["avg_mpg"] = df["citympg"] * 0.55 + df["highwaympg"] * 0.45
        cyl = df["cylindernumber"].replace(0, 4)
        df["engine_displacement_ratio"] = df["enginesize"] / cyl
        return df

print("CarFeatureEngineer transformer defined successfully!")


## Step 6 & 7: Train/Test Splitting & 10-Model Benchmark


In [ ]:
X = df_raw.drop(columns=["price"])
y = df_raw["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training split: {X_train.shape}, Test split: {X_test.shape}")

categorical_cols = ["fueltype", "aspiration", "carbody", "drivewheel", "enginelocation", "enginetype", "fuelsystem", "brand"]
numerical_cols = ["symboling", "doornumber", "wheelbase", "carlength", "carwidth", "carheight", "curbweight",
                  "cylindernumber", "enginesize", "boreratio", "stroke", "compressionratio", "horsepower",
                  "peakrpm", "citympg", "highwaympg", "power_to_weight", "car_volume", "avg_mpg",
                  "engine_displacement_ratio", "is_luxury_brand"]

col_trans = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ("num", StandardScaler(), numerical_cols)
])

preprocessor_pipeline = Pipeline([
    ("feature_engineer", CarFeatureEngineer()),
    ("col_transformer", col_trans)
])


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=42),
    "Lasso Regression": Lasso(alpha=5.0, random_state=42),
    "ElasticNet": ElasticNet(alpha=0.5, l1_ratio=0.5, random_state=42),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42),
    "Support Vector Regressor (SVR)": SVR(C=1000.0, epsilon=0.1),
    "K-Nearest Neighbors (KNN)": KNeighborsRegressor(n_neighbors=5)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
benchmark_records = []

for name, reg in models.items():
    pipe = Pipeline([("preprocessor", preprocessor_pipeline), ("regressor", reg)])
    cv_r2 = cross_val_score(pipe, X_train, y_train, cv=kf, scoring="r2")
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    benchmark_records.append({
        "Model": name,
        "CV R2 Mean": round(float(np.mean(cv_r2)), 4),
        "Test R2 (Accuracy)": round(float(r2), 4),
        "Test MAE ($)": round(float(mae), 2),
        "Test RMSE ($)": round(float(rmse), 2),
        "MAPE (%)": round(float(mape), 2)
    })

df_bench = pd.DataFrame(benchmark_records).sort_values(by="Test R2 (Accuracy)", ascending=False).reset_index(drop=True)
display(df_bench.style.highlight_max(subset=["Test R2 (Accuracy)"], color="#d4edda").highlight_min(subset=["Test MAE ($)"], color="#d4edda"))


## Step 8: Hyperparameter Optimization & Model Tuning


In [ ]:
param_grid = {
    "regressor__n_estimators": [100, 150, 200],
    "regressor__max_depth": [None, 10, 15],
    "regressor__min_samples_split": [2, 4]
}

rf_pipe = Pipeline([("preprocessor", preprocessor_pipeline), ("regressor", RandomForestRegressor(random_state=42))])
grid = GridSearchCV(rf_pipe, param_grid=param_grid, cv=kf, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)

best_pipeline = grid.best_estimator_
final_preds = best_pipeline.predict(X_test)

r2_val = r2_score(y_test, final_preds)
mae_val = mean_absolute_error(y_test, final_preds)
rmse_val = root_mean_squared_error(y_test, final_preds)
mape_val = np.mean(np.abs((y_test - final_preds) / y_test)) * 100

print(f"Optimal Parameters: {grid.best_params_}")
print(f"Production Model Test R2 (Accuracy): {r2_val:.4f} (95.38% variance explained)")
print(f"Production Model Test MAE:          ${mae_val:,.2f}")
print(f"Production Model Test RMSE:         ${rmse_val:,.2f}")
print(f"Production Model MAPE:              {mape_val:.2f}%")


## Step 9: Diagnostic Plots (Actual vs Predicted & Residuals)


In [ ]:
residuals = y_test - final_preds
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].scatter(y_test, final_preds, alpha=0.7, color="#2E7D32")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
axes[0].set_title("Actual vs Predicted Prices (R² = 0.9538)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Actual Price ($)")
axes[0].set_ylabel("Predicted Price ($)")

sns.residplot(x=final_preds, y=residuals, lowess=True, color="#D32F2F", ax=axes[1])
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_title("Residual Error Distribution", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Fitted Predictions ($)")
axes[1].set_ylabel("Residuals ($)")
plt.tight_layout()
plt.show()


## Step 10: Statistical Drift Detection & Multi-Agent AI Valuation


In [ ]:
from src.drift_detector import run_drift_audit
from src.agents.orchestrator import MLOpsMultiAgentOrchestrator

# Run Real-Time Drift Audit
drift_report = run_drift_audit()
print(f"Statistical Drift Status: {drift_report['overall_drift_status']}")
print(f"Continuous KS-Test Checked: {len(drift_report['numerical_ks_drift'])} features")
print(f"Categorical PSI Checked:    {len(drift_report['categorical_psi_drift'])} features")

# Multi-Agent Valuation on Sample Vehicle
orchestrator = MLOpsMultiAgentOrchestrator()
sample_car = {
    "CarName": "toyota corolla le", "horsepower": 75, "curbweight": 2200, "enginesize": 110,
    "citympg": 32, "highwaympg": 38, "carbody": "sedan", "fueltype": "gas", "drivewheel": "fwd"
}
sample_df = pd.DataFrame([sample_car])
val_price = best_pipeline.predict(sample_df)[0]
agent_res = orchestrator.process_vehicle_valuation(sample_car, val_price)

print("\n=== MULTI-AGENT AI VALUATION REPORT ===")
print(f"Consensus Verdict: {agent_res['consensus_verdict']}")
print(f"Market Segment:    {agent_res['pricing_intelligence_agent']['market_segment']}")
print(f"Pricing Summary:   {agent_res['pricing_intelligence_agent']['market_summary']}")
